In [80]:
import json
import networkx as nx

with open("crime_kg_nodes_edges.json", "r", encoding="utf-8") as f:
    kg_data = json.load(f)

G = nx.Graph()

for node in kg_data["nodes"]:
    node_id = node["id"]
    attrs = {k: v for k, v in node.items() if k != "id"}
    G.add_node(node_id, **attrs)

for edge in kg_data["edges"]:
    G.add_edge(edge["source"], edge["target"], relationship=edge["relationship"])

print("Nodes:", G.number_of_nodes(), "Edges:", G.number_of_edges())

Nodes: 1530 Edges: 2127


In [68]:
relationship_types = {
    "PERPETRATED_BY": ("CASE", "SUSPECT"),
    "OFFENSE_TYPE": ("CASE", "CRIME_TYPE"),
    "OCCURRED_IN_BEAT": ("CASE", "POLICE_BEAT"),
    "OCCURRED_AT": ("CASE", "LOCATION"),
    "DRIVES_VEHICLE": ("SUSPECT", "VEHICLE"),
}

from collections import Counter
print("Node types:", Counter(d.get("type") for _, d in G.nodes(data=True)))
print("Relationships:", Counter(d.get("relationship") for _, _, d in G.edges(data=True)))

Node types: Counter({'CASE': 500, 'LOCATION': 462, 'SUSPECT': 434, 'VEHICLE': 104, 'POLICE_BEAT': 22, 'CRIME_TYPE': 8})
Relationships: Counter({'PERPETRATED_BY': 500, 'OFFENSE_TYPE': 500, 'OCCURRED_IN_BEAT': 500, 'OCCURRED_AT': 500, 'DRIVES_VEHICLE': 127})


In [69]:
from sklearn.model_selection import train_test_split

all_edges = list(G.edges(data=True))

train_edges, test_edges = train_test_split(
    all_edges, test_size=0.20, random_state=42
)

G_train = nx.Graph()
G_train.add_nodes_from(G.nodes(data=True))
G_train.add_edges_from(train_edges)

print("Full graph edges:", G.number_of_edges())
print("Train graph edges:", G_train.number_of_edges())
print("Held-out test edges:", len(test_edges))

Full graph edges: 2127
Train graph edges: 1701
Held-out test edges: 426


In [70]:
import random
random.seed(42)

existing_edges = {frozenset((u, v)) for u, v in G.edges()}

test_positive = []
for source, target, data in test_edges:
    test_positive.append((source, target, data["relationship"]))

test_negative = []
for source, target, relationship in test_positive:
    type1 = G.nodes[source]["type"]
    type2 = G.nodes[target]["type"]

    candidates1 = [n for n, d in G.nodes(data=True) if d.get("type") == type1]
    candidates2 = [n for n, d in G.nodes(data=True) if d.get("type") == type2]

    while True:
        n1 = random.choice(candidates1)
        n2 = random.choice(candidates2)
        if n1 != n2 and frozenset((n1, n2)) not in existing_edges:
            test_negative.append((n1, n2, relationship))
            break

print("Test positives:", len(test_positive))
print("Test negatives:", len(test_negative))

Test positives: 426
Test negatives: 426


In [71]:
train_positive = []
for source, target, data in train_edges:
    train_positive.append((source, target, data["relationship"]))

train_negative = []
for source, target, relationship in train_positive:
    type1 = G.nodes[source]["type"]
    type2 = G.nodes[target]["type"]

    candidates1 = [n for n, d in G.nodes(data=True) if d.get("type") == type1]
    candidates2 = [n for n, d in G.nodes(data=True) if d.get("type") == type2]

    while True:
        n1 = random.choice(candidates1)
        n2 = random.choice(candidates2)
        if n1 != n2 and frozenset((n1, n2)) not in existing_edges:
            train_negative.append((n1, n2, relationship))
            break

print("Train positives:", len(train_positive))
print("Train negatives:", len(train_negative))

Train positives: 1701
Train negatives: 1701


In [72]:
import numpy as np

ALL_NODE_TYPES = ["CASE", "SUSPECT", "CRIME_TYPE", "POLICE_BEAT", "LOCATION", "VEHICLE"]
ALL_RELATIONSHIPS = ["PERPETRATED_BY", "OFFENSE_TYPE", "OCCURRED_IN_BEAT", "OCCURRED_AT", "DRIVES_VEHICLE"]

def get_pair_features(G, node1, node2, relationship):
    degree1 = G.degree(node1)
    degree2 = G.degree(node2)

    neighbors1 = set(G.neighbors(node1))
    neighbors2 = set(G.neighbors(node2))
    shared = neighbors1 & neighbors2
    union = neighbors1 | neighbors2

    common_neighbors = len(shared)
    jaccard = common_neighbors / len(union) if union else 0.0
    degree_sum = degree1 + degree2
    degree_diff = abs(degree1 - degree2)
    degree_ratio = min(degree1, degree2) / max(degree1, degree2) if max(degree1, degree2) > 0 else 0.0

    adamic_adar = 0.0
    resource_allocation = 0.0
    for n in shared:
        nd = G.degree(n)
        if nd > 1:
            adamic_adar += 1 / np.log(nd)
        if nd > 0:
            resource_allocation += 1 / nd

    shared_types = {}
    for n in shared:
        t = G.nodes[n].get("type")
        shared_types[t] = shared_types.get(t, 0) + 1

    features = [degree1, degree2, common_neighbors, jaccard, degree_sum,
                degree_diff, degree_ratio, adamic_adar, resource_allocation]

    for t in ALL_NODE_TYPES:
        features.append(shared_types.get(t, 0))

    for rel in ALL_RELATIONSHIPS:
        features.append(1 if relationship == rel else 0)

    return features

FEATURE_NAMES = (
    ["degree1", "degree2", "common_neighbors", "jaccard", "degree_sum",
     "degree_diff", "degree_ratio", "adamic_adar", "resource_allocation"]
    + [f"shared_{t}" for t in ALL_NODE_TYPES]
    + [f"rel_{r}" for r in ALL_RELATIONSHIPS]
)

print("Num features:", len(FEATURE_NAMES))

Num features: 20


In [73]:
X_train = []
y_train = []

for node1, node2, relationship in train_positive:
    G_temp = G_train.copy()
    if G_temp.has_edge(node1, node2):
        G_temp.remove_edge(node1, node2)
    X_train.append(get_pair_features(G_temp, node1, node2, relationship))
    y_train.append(1)

for node1, node2, relationship in train_negative:
    X_train.append(get_pair_features(G_train, node1, node2, relationship))
    y_train.append(0)

X_train = np.array(X_train, dtype=float)
y_train = np.array(y_train, dtype=int)

print("X_train:", X_train.shape, "Positives:", (y_train==1).sum(), "Negatives:", (y_train==0).sum())

X_train: (3402, 20) Positives: 1701 Negatives: 1701


In [74]:
X_test = []
y_test = []

for node1, node2, relationship in test_positive:
    X_test.append(get_pair_features(G_train, node1, node2, relationship))
    y_test.append(1)

for node1, node2, relationship in test_negative:
    X_test.append(get_pair_features(G_train, node1, node2, relationship))
    y_test.append(0)

X_test = np.array(X_test, dtype=float)
y_test = np.array(y_test, dtype=int)

print("X_test:", X_test.shape, "Positives:", (y_test==1).sum(), "Negatives:", (y_test==0).sum())

X_test: (852, 20) Positives: 426 Negatives: 426


In [75]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix
)

model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("OVERALL RESULTS")
print("Accuracy :", round(accuracy_score(y_test, y_pred), 4))
print("Precision:", round(precision_score(y_test, y_pred), 4))
print("Recall   :", round(recall_score(y_test, y_pred), 4))
print("F1 Score :", round(f1_score(y_test, y_pred), 4))
print("ROC-AUC  :", round(roc_auc_score(y_test, y_prob), 4))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

OVERALL RESULTS
Accuracy : 0.7371
Precision: 0.8042
Recall   : 0.6268
F1 Score : 0.7045
ROC-AUC  : 0.8141

Confusion Matrix:
[[361  65]
 [159 267]]


In [76]:
print("PER-RELATIONSHIP PERFORMANCE")
print("=" * 40)

all_test = test_positive + test_negative
all_labels = np.concatenate([np.ones(len(test_positive)), np.zeros(len(test_negative))]).astype(int)

for rel in ALL_RELATIONSHIPS:
    mask = np.array([item[2] == rel for item in all_test])
    if mask.sum() == 0:
        continue
    y_true_r = y_test[mask]
    y_pred_r = y_pred[mask]
    y_prob_r = y_prob[mask]

    print(f"\n{rel}  (n={mask.sum()})")
    print("  Accuracy :", round(accuracy_score(y_true_r, y_pred_r), 4))
    print("  Precision:", round(precision_score(y_true_r, y_pred_r, zero_division=0), 4))
    print("  Recall   :", round(recall_score(y_true_r, y_pred_r, zero_division=0), 4))
    print("  F1 Score :", round(f1_score(y_true_r, y_pred_r, zero_division=0), 4))
    if len(set(y_true_r)) > 1:
        print("  ROC-AUC  :", round(roc_auc_score(y_true_r, y_prob_r), 4))

PER-RELATIONSHIP PERFORMANCE

PERPETRATED_BY  (n=194)
  Accuracy : 0.7629
  Precision: 0.84
  Recall   : 0.6495
  F1 Score : 0.7326
  ROC-AUC  : 0.7538

OFFENSE_TYPE  (n=192)
  Accuracy : 0.625
  Precision: 0.7
  Recall   : 0.4375
  F1 Score : 0.5385
  ROC-AUC  : 0.7479

OCCURRED_IN_BEAT  (n=204)
  Accuracy : 0.6618
  Precision: 0.7619
  Recall   : 0.4706
  F1 Score : 0.5818
  ROC-AUC  : 0.7876

OCCURRED_AT  (n=210)
  Accuracy : 0.881
  Precision: 0.8704
  Recall   : 0.8952
  F1 Score : 0.8826
  ROC-AUC  : 0.8704

DRIVES_VEHICLE  (n=52)
  Accuracy : 0.7692
  Precision: 0.7692
  Recall   : 0.7692
  F1 Score : 0.7692
  ROC-AUC  : 0.7692


In [77]:
print("MODEL COEFFICIENTS")
print("=" * 40)
for name, coef in zip(FEATURE_NAMES, model.coef_[0]):
    print(f"{name:22s}: {coef:.4f}")

MODEL COEFFICIENTS
degree1               : -0.2603
degree2               : -0.2657
common_neighbors      : 0.0000
jaccard               : 0.0000
degree_sum            : -0.5260
degree_diff           : 0.7756
degree_ratio          : -0.3374
adamic_adar           : 0.0000
resource_allocation   : 0.0000
shared_CASE           : 0.0000
shared_SUSPECT        : 0.0000
shared_CRIME_TYPE     : 0.0000
shared_POLICE_BEAT    : 0.0000
shared_LOCATION       : 0.0000
shared_VEHICLE        : 0.0000
rel_PERPETRATED_BY    : -1.2141
rel_OFFENSE_TYPE      : 2.4090
rel_OCCURRED_IN_BEAT  : 2.1225
rel_OCCURRED_AT       : -1.6813
rel_DRIVES_VEHICLE    : -1.6387


In [78]:
import joblib
joblib.dump(model, "criminal_network_link_prediction_final.pkl")
print("Model saved.")

Model saved.


In [79]:
def predict_link(node1, node2, relationship, G):
    features = np.array(get_pair_features(G, node1, node2, relationship)).reshape(1, -1)
    probability = model.predict_proba(features)[0][1]
    prediction = model.predict(features)[0]
    return {
        "node1": node1,
        "node2": node2,
        "relationship": relationship,
        "link_probability": round(float(probability), 4),
        "predicted_link": int(prediction)
    }

# quick sanity check on a real held-out edge
example = test_positive[0]
print(predict_link(example[0], example[1], example[2], G_train))

{'node1': 'Beat 0412', 'node2': 'JC112204', 'relationship': 'OCCURRED_IN_BEAT', 'link_probability': 0.4334, 'predicted_link': 0}


In [83]:
import json

top_predictions = sorted(candidate_predictions, key=lambda x: x["link_probability"], reverse=True)[:50]
with open("link_predictions_output.json", "w") as f:
    json.dump(top_predictions, f, indent=2)
print("Saved link_predictions_output.json")

Saved link_predictions_output.json
